# Deduplicating Donors Across Vendor Lists

**Scenario:** A political campaign receives donor lists from three vendors.
Each vendor formats names differently — ALL CAPS, mixed case, abbreviations.
We need to match donors across lists without manual review.

siege_utilities solves this with three primitives:
1. **Name normalization** — canonicalize case, punctuation, and whitespace so "John Smith" and "john smith" produce the same seed
2. **Deterministic namespaces** — derive org-specific UUID namespaces
3. **Deterministic UUIDs** — same normalized input always produces the same UUID

**Limitation:** `normalize_name_v1` handles case, punctuation, and whitespace
but does NOT reorder names. "SMITH, JOHN A" (Last, First) normalizes to
"smith john a", not "john smith". Matching across name-order conventions
requires preprocessing before normalization — a v2 concern.

## What this shows

Deduplicate donor/customer-style records across vendor lists using normalized names, deterministic namespaces, and UUID5 identifiers. This is a pure/offline foundations notebook: it should run without network, credentials, GDAL, Spark, or external services.


## The Problem: Three Vendors, Three Conventions

Vendor A sends ALL CAPS. Vendor B uses mixed case with middle initials.
Vendor C uses first-name-only with abbreviations. A human can see these
are the same person — but how do we match them programmatically?

In [ ]:
from siege_utilities.identifiers.normalize import normalize_name_v1

# Three vendors, same donors, different formatting
vendor_a = ["SMITH, JOHN A", "GARCIA, MARIA L", "O'BRIEN, PATRICK"]
vendor_b = ["John Smith", "Maria L. Garcia", "Patrick O'Brien"]
vendor_c = ["john smith", "maria garcia", "pat obrien"]

# Normalize everything to a canonical form
print("Vendor A normalized:")
for name in vendor_a:
    print(f"  {name:25s} → {normalize_name_v1(name)}")

print("\nVendor B normalized:")
for name in vendor_b:
    print(f"  {name:25s} → {normalize_name_v1(name)}")

print("\nVendor C normalized:")
for name in vendor_c:
    print(f"  {name:25s} → {normalize_name_v1(name)}")

## Deterministic UUIDs for Cross-List Matching

Normalized names give us a matching key, but we need stable identifiers
that survive across databases and systems. UUID5 (SHA-1 based) is
deterministic: same namespace + same input = same UUID, every time.

In [ ]:
from siege_utilities.identifiers.namespaces import derive_root, derive_sub_namespace
from siege_utilities.identifiers.uuid_generation import uuid5_from_seed

# Organization namespace — unique to our campaign
campaign_ns = derive_root("tx32-campaign-2024")
donor_ns = derive_sub_namespace(campaign_ns, "donor")
print(f"Campaign namespace: {campaign_ns}")
print(f"Donor namespace:    {donor_ns}")

# Generate UUIDs from normalized names
all_vendors = {
    "Vendor A": ["SMITH, JOHN A", "GARCIA, MARIA L", "O'BRIEN, PATRICK"],
    "Vendor B": ["John Smith", "Maria L. Garcia", "Patrick O'Brien"],
    "Vendor C": ["john smith", "maria garcia", "pat obrien"],
}

print("\nDonor UUID mapping:")
print(f"{'Vendor':<10} {'Original':<25} {'Normalized':<20} {'UUID'}")
print("-" * 90)
for vendor, names in all_vendors.items():
    for name in names:
        normalized = normalize_name_v1(name)
        uid = uuid5_from_seed(donor_ns, normalized)
        print(f"{vendor:<10} {name:<25} {normalized:<20} {uid}")

## Verifying Determinism

The key property: running this tomorrow, on a different machine, with the
same inputs, produces the same UUIDs. No database sequence, no random seed.

Note that "SMITH, JOHN A" and "John Smith" produce *different* UUIDs below
because `normalize_name_v1` preserves word order — "smith john a" ≠ "john smith".
Names in the same format (like "John Smith" and "john smith") DO match.

In [ ]:
# Same input, same output — always
uid_1 = uuid5_from_seed(donor_ns, normalize_name_v1("SMITH, JOHN A"))
uid_2 = uuid5_from_seed(donor_ns, normalize_name_v1("John Smith"))
uid_3 = uuid5_from_seed(donor_ns, normalize_name_v1("john smith"))

print(f"From 'SMITH, JOHN A':  {uid_1}")
print(f"From 'John Smith':     {uid_2}")
print(f"From 'john smith':     {uid_3}")
print(f"\nAll match: {uid_1 == uid_2 == uid_3}")

# Different person, different namespace = different UUID
precinct_ns = derive_sub_namespace(campaign_ns, "precinct")
uid_donor = uuid5_from_seed(donor_ns, "smith john")
uid_precinct = uuid5_from_seed(precinct_ns, "smith john")
print(f"\nSame seed, donor namespace:    {uid_donor}")
print(f"Same seed, precinct namespace: {uid_precinct}")
print(f"Namespaces prevent collisions: {uid_donor != uid_precinct}")

## Key Takeaways

- **normalize_name_v1** strips punctuation, lowercases, and collapses whitespace (preserves word order — does NOT reorder "Last, First" to "first last")
- **derive_root / derive_sub_namespace** create organization-scoped UUID namespaces
- **uuid5_from_seed** generates deterministic UUIDs: same namespace + same input = same UUID
- Combine all three to match entities across vendor lists where formatting differs but word order is consistent
- Matching across name-order conventions (e.g., "SMITH, JOHN" vs "John Smith") requires preprocessing before normalization

## Related

- Source: `siege_utilities/identifiers/normalize.py`, `siege_utilities/identifiers/namespaces.py`, `siege_utilities/identifiers/uuid_generation.py`
- Tests: `tests/test_identifiers_namespaces_errors.py`, `tests/test_identifiers_uuid_generation_errors.py`
- Notebook governance: `tests/test_notebook_hygiene.py`, `tests/test_notebooks.py`, `scripts/check_notebook_inventory.py`
